### What are the types of joins and joins strategies in spark?

In Spark (PySpark / Spark SQL), interviewers typically ask this expecting an answer across two layers: **Logical Join Types** (the SQL syntax/semantic intent) and **Physical Join Strategies** (the execution algorithms under the Catalyst optimizer).

---

#### 1. Logical Join Types (Semantic Joins)

* **`inner` (Default):** Returns rows with matching keys in both DataFrames.
* **`left` / `left_outer`:** Returns all rows from the left DataFrame and matching values from the right (or `NULL` if no match).
* **`right` / `right_outer`:** Returns all rows from the right DataFrame and matching values from the left (or `NULL` if no match).
* **`full` / `full_outer`:** Returns all rows when there is a match in either the left or right side, filling mismatches with `NULL`.
* **`left_semi` / `semi`:** Returns only the columns and rows from the left DataFrame where a matching key exists in the right (acts like an `EXISTS` subquery; does not duplicate left rows on 1-to-many matches).
* **`left_anti` / `anti`:** Returns only the columns and rows from the left DataFrame that have **no** matching key in the right (acts like a `NOT EXISTS` subquery).
* **`cross`:** Computes the full Cartesian product ($M \times N$ rows). In newer versions, non-equi joins without explicit conditions trigger this or require `spark.sql.crossJoin.enabled=true`.

---

#### 2. Physical Join Strategies (Engine Execution Under the Hood)

Spark Catalyst selects from **5 physical join strategies** ranked by cost and dataset size:

| Strategy | When Spark Uses It | Key Characteristic / Advantage |
| --- | --- | --- |
| **Broadcast Hash Join (BHJ)** | One side is smaller than `spark.sql.autoBroadcastJoinThreshold` (default 10 MB) or forced via `broadcast()`. | **No shuffle.** Driver broadcasts the small table to all executor memory. Fastest join. |
| **Shuffle Hash Join (SHJ)** | One side is relatively smaller than the other (builds hash table per partition) and `preferSortMergeJoin` is false or AQE dynamically demotes SMJ. | Shuffles by key, builds in-memory hash table on small side per partition. Avoids sorting overhead. |
| **Shuffle Sort Merge Join (SMJ)** | Default for two large DataFrames joining on equi-join keys (`=`). | Shuffles both tables on join keys, **sorts** partitions, then merges iteratively. Highly robust against Out-Of-Memory (OOM) errors. |
| **Broadcast Nested Loop Join (BNLJ)** | Used for non-equi joins (`<`, `>`, `!=`) or when no join key is provided and one side can be broadcast. | Broadcasts one side and loops over every row of the other. Can be slow if datasets grow. |
| **Cartesian Product Join (CPJ)** | Full shuffle nested loop used for large-scale cross-joins without equi-conditions. | Shuffles all partitions across all executors. Heaviest compute cost. |

---

#### Key Takeaway for the Interview

- If asked about performance tuning: mention that **Adaptive Query Execution (AQE)** at runtime dynamically converts **Sort Merge Join (SMJ) into Broadcast Hash Join (BHJ)** if post-filter statistics show a table dropped below the broadcast threshold.

### How do you handle severe data skew during joins in PySpark and Databricks?

Severe data skew occurs when a specific join key is overwhelmingly frequent (e.g., `null`, default, or high-volume category IDs), causing most records to route to a single partition. This creates the classic **"99% completed task hangs on the last task"** executor bottleneck or triggers an Out-Of-Memory (OOM) error.

Here is how to tackle it in an interview setting, structured from **automatic runtime features** to **manual code-level techniques**.

---

#### 1. Enable Adaptive Query Execution (AQE) Skew Join Handling

In Databricks and modern Spark (3.0+), AQE is enabled by default. It detects skew partitions at runtime post-shuffle and automatically splits the skewed partition into smaller sub-partitions.

**Configuration:**
*   ```python
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
    # Default skew thresholds (tweak if Spark fails to flag the partition as skewed):
    spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", 5) # 5x median partition size
    spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "64MB")

    ```
---

#### 2. Manual Salting (The Gold-Standard Coding Answer)

If AQE cannot resolve the skew or you need programmatic control, use **salting** to distribute the hot key across multiple partitions.

##### Step-by-Step Logic:

1. On the **skewed (large) DataFrame**, append a random integer column (`salt`) between `0` and `N - 1` (e.g., $N = 5$).
2. On the **lookup/dimension DataFrame**, explode the rows using `array([lit(i) for i in range(N)])` so every lookup row duplicates with salt values `0` through `N - 1`.
3. Perform the join on `['join_key', 'salt']`.
4. Drop the `salt` column afterward.

**Code**

*   ```python
    from pyspark.sql import functions as F

    salt_factor = 5

    # 1. Add random salt to the skewed table
    df_large_salted = df_large.withColumn("salt", (F.rand() * salt_factor).cast("int"))

    # 2. Explode the smaller/lookup table to match all possible salt keys
    df_small_salted = df_small.withColumn("salt_array", F.array([F.lit(i) for i in range(salt_factor)])) \
                            .withColumn("salt", F.explode("salt_array")) \
                            .drop("salt_array")

    # 3. Join on both the original key and the salt
    df_joined = df_large_salted.join(df_small_salted, on=["id", "salt"], how="inner").drop("salt")

    ```
---

#### 3. Broadcast Hash Join (BHJ)

If one of the datasets is small enough to fit in executor memory:

* Eliminate the shuffle entirely by broadcasting the small table:
* ```python
    from pyspark.sql.functions import broadcast
    df_joined = df_large.join(broadcast(df_small), on="id", how="inner")

  ```
* Because all executors receive a full copy of the small table, no shuffle step happens, completely neutralizing key skew.

---

#### 4. Split-and-Union Pattern (Isolating Hot Keys)

If only a handful of specific keys (like `NULL` or `'UNKNOWN'`) cause 90% of the skew:

1. Filter the dataset into two subsets: `df_skewed` (only hot keys) and `df_unskewed`.
2. Handle the `df_unskewed` with a standard Sort-Merge / Shuffle Hash join.
3. Handle `df_skewed` separately (e.g., broadcast join, salting, or null-handling logic).
4. `unionByName()` both results.

---

#### 5. Databricks-Specific Optimizations (Delta Lake)

* **Z-Ordering / Liquid Clustering:** If queries frequently join or filter on the skewed key, apply Delta Liquid Clustering (`CLUSTER BY (join_key)`) or Z-Order to minimize data scanning and improve partition skipping.
* **Join Hints:** Use explicit Databricks skew hints if you know the exact skewed key:
    *   ```sql
        SELECT /*+ SKEW('df_large', 'id', ('NULL', 101, 102)) */ *
        FROM df_large
        JOIN df_small ON df_large.id = df_small.id

        ```